# Assignment 10: MT19937 Stream Cipher and Predictable Token Attack

## Part A: MT19937-Based Stream Cipher and Seed Recovery

### Objective
- Understand how MT19937 can be used to generate a pseudorandom keystream.
- Implement a toy stream cipher using XOR encryption.
- Verify encryption and decryption using the same seed.
- Exploit known plaintext consisting of 14 consecutive `A` characters.
- Brute-force all 65,536 possible 16-bit seeds to recover the original seed.
- Verify the recovered seed by decrypting the complete ciphertext.

## 1. Understanding MT19937

MT19937 (Mersenne Twister) is a pseudorandom number generator (PRNG).

Key characteristics:
- It has a state size of 624 words, with each word containing 32 bits.
- It uses a recurrence relation to update its internal state.
- It applies a tempering operation to produce output values.
- It has a period of \(2^{19937}-1\).
- It produces statistically good pseudorandom sequences.

### Why is MT19937 not cryptographically secure?

Despite its good statistical properties, MT19937 is not designed for cryptographic security.

- Its internal state can be reconstructed from sufficient observed outputs.
- Its output is deterministic when the seed and state are known.
- It is unsuitable for generating cryptographic keys, secure tokens, or encryption keystreams.

In this assignment, we demonstrate how a small seed space makes brute-force recovery practical.

## 2. MT19937-Based Stream Cipher

A stream cipher encrypts data by combining plaintext with a generated keystream.

In this assignment:
1. MT19937 generates pseudorandom bytes using a seed.
2. Each keystream byte is XORed with the corresponding plaintext byte.
3. The resulting bytes form the ciphertext.

### Encryption

\[
C_i = P_i \oplus K_i
\]

Where:
- \(P_i\) = plaintext byte
- \(K_i\) = keystream byte
- \(C_i\) = ciphertext byte

### Decryption

\[
P_i = C_i \oplus K_i
\]

XOR is reversible because:

\[
(A \oplus B) \oplus B = A
\]

Therefore, the same function can perform both encryption and decryption, provided the same seed and keystream generation procedure are used.

## 3. Known Plaintext Attack

A known plaintext attack occurs when an attacker knows some part of the original plaintext and has access to its corresponding ciphertext.

For this assignment:
- The attacker knows that the plaintext contains 14 consecutive `A` characters.
- The location of this sequence is unknown.
- The encryption seed is a randomly selected 16-bit integer.

### How can the known plaintext help?

Since:

\[
C_i = P_i \oplus K_i
\]

We can calculate the corresponding keystream bytes:

\[
K_i = C_i \oplus P_i
\]

For the known sequence:

\[
K_i = C_i \oplus \texttt{0x41}
\]

The attacker can use this information to identify the correct seed by testing possible seeds and checking whether the decrypted plaintext contains 14 consecutive `A` characters.

### Search space

A 16-bit seed has:

\[
2^{16} = 65,536
\]

possible values.

Therefore, the attacker can test every possible seed instead of needing to guess the original one.

In [1]:
import numpy as np
import secrets
import string


def generate_keystream(seed: int, length: int) -> bytes:
    bit_generator = np.random.MT19937(seed)
    rng = np.random.Generator(bit_generator)

    return rng.integers(
        0, 256, size=length, dtype=np.uint8
    ).tobytes()


def xor_cipher(data: bytes, seed: int) -> bytes:
    keystream = generate_keystream(seed, len(data))
    return bytes(a ^ b for a, b in zip(data, keystream))

In [2]:
# Generate a random prefix of 5–20 printable characters
prefix_length = secrets.randbelow(16) + 5

printable_chars = (
    string.ascii_letters + string.digits + string.punctuation + " "
)

prefix = ''.join(
    secrets.choice(printable_chars)
    for _ in range(prefix_length)
)

# Append 14 consecutive A characters
plaintext = (prefix + "A" * 14).encode("utf-8")

# Generate a random 16-bit seed
original_seed = secrets.randbelow(65536)

# Encrypt the plaintext
ciphertext = xor_cipher(plaintext, original_seed)

print("Original plaintext:", plaintext.decode())
print("Original seed:", original_seed)
print("Plaintext length:", len(plaintext), "bytes")
print("Ciphertext (hex):", ciphertext.hex())

Original plaintext: 0-#x-RDRe9#9AAAAAAAAAAAAAA
Original seed: 54742
Plaintext length: 26 bytes
Ciphertext (hex): e37b7538bdf5ed9ebc00101fc2d8a8fe97fe69673d01ba18d9e4


In [3]:
def recover_seed(ciphertext: bytes):
    known_sequence = b"A" * 14
    matching_candidates = []

    for candidate_seed in range(65536):
        decrypted = xor_cipher(ciphertext, candidate_seed)

        if known_sequence in decrypted:
            matching_candidates.append(
                (candidate_seed, decrypted)
            )

    return matching_candidates


matches = recover_seed(ciphertext)

print("Total seeds tested:", 65536)
print("Matching candidates:", len(matches))

for seed, recovered_plaintext in matches:
    print("Recovered seed:", seed)
    print("Recovered plaintext:", recovered_plaintext.decode())

Total seeds tested: 65536
Matching candidates: 1
Recovered seed: 54742
Recovered plaintext: 0-#x-RDRe9#9AAAAAAAAAAAAAA


In [4]:
recovered_seed, recovered_plaintext = matches[0]

# Decrypt the complete ciphertext using the recovered seed
verified_plaintext = xor_cipher(ciphertext, recovered_seed)

print("Original seed:", original_seed)
print("Recovered seed:", recovered_seed)
print("Seed recovery successful:", recovered_seed == original_seed)

print("\nOriginal plaintext:", plaintext.decode())
print("Recovered plaintext:", verified_plaintext.decode())

print("\nPlaintexts match:", verified_plaintext == plaintext)
print("14 consecutive A's found:", b"A" * 14 in verified_plaintext)
print("Complete ciphertext decrypted:", len(verified_plaintext) == len(ciphertext))

Original seed: 54742
Recovered seed: 54742
Seed recovery successful: True

Original plaintext: 0-#x-RDRe9#9AAAAAAAAAAAAAA
Recovered plaintext: 0-#x-RDRe9#9AAAAAAAAAAAAAA

Plaintexts match: True
14 consecutive A's found: True
Complete ciphertext decrypted: True


| Parameter | Result |
|---|---|
| Original seed | 54742 |
| Seed size | 16 bits |
| Total candidate seeds tested | 65,536 |
| Matching candidates | 1 |
| Recovered seed | 54742 |
| Known plaintext sequence | 14 consecutive A characters |
| Plaintext recovery | Successful |
| Complete ciphertext decryption | Successful |

# Part B: Predictable Password Reset Tokens

## Objective

To demonstrate how a password reset token generated using MT19937 with a timestamp as its seed can be predicted through brute-force searching.

## Scenario

A simulated application generates a 16-byte password reset token using MT19937. The generator is seeded with the current Unix timestamp, represented as an integer in seconds.

An attacker who knows the approximate token generation time can search a limited range of timestamps and reproduce the token.

## Steps Performed

1. Record the Unix timestamp at token generation.
2. Initialize MT19937 using the timestamp as the seed.
3. Generate 16 consecutive 8-bit values.
4. Combine these values into a 16-byte token.
5. Display the timestamp and token in hexadecimal.
6. Search candidate timestamps within T - 300 to T + 300 seconds.
7. Compare each generated candidate token with the observed token.
8. Regenerate the token using the recovered timestamp to verify the result.

## Why Timestamp-Based Seeding Is Vulnerable

MT19937 is a deterministic pseudorandom number generator. This means that the same seed always produces the same sequence of pseudorandom values.

In this experiment, the Unix timestamp (integer seconds) is used as the seed for generating the reset token.

This creates a security weakness because:

1. Unix timestamps are not secret and can often be estimated by an attacker.
2. The timestamp seed has a predictable value within a limited time interval.
3. MT19937 produces the same output sequence whenever it is initialized with the same seed.
4. An attacker who knows the approximate token generation time can test possible timestamps.
5. If a generated candidate token matches the observed token, the attacker has recovered the seed and can reproduce the token.

### Security Implications

- Reset tokens become predictable if their generation time can be estimated.
- A large search space is not required when the timestamp interval is small.
- The deterministic nature of MT19937 makes token reproduction possible.
- Password reset tokens should never depend on predictable timestamps or non-cryptographic PRNGs.

### Recommended Secure Approach

Applications should use a cryptographically secure random number generator (CSPRNG) to generate password reset tokens.

For example, Python provides the `secrets` module, which is designed for security-sensitive random values.

Therefore, MT19937 should not be used for password reset tokens, authentication tokens, cryptographic keys, or other security-sensitive values.

## Pseudocode

### Part A: MT19937 Known-Plaintext Seed Recovery

```text
INPUT: Ciphertext
KNOWN_SEQUENCE = 14 consecutive 'A' characters

FOR seed = 0 TO 65535:
    Initialize MT19937 using seed
    Generate keystream equal to ciphertext length
    Decrypt ciphertext using XOR

    IF decrypted plaintext contains KNOWN_SEQUENCE:
        Record seed and decrypted plaintext

OUTPUT:
    Recovered seed
    Recovered plaintext
    Verify complete plaintext

INPUT: Observed 16-byte reset token
       Approximate timestamp T
       Search window = 300 seconds

START = T - 300
END = T + 300

FOR timestamp = START TO END:
    Initialize MT19937 using timestamp
    Generate 16 consecutive 8-bit values
    Combine values into a 16-byte candidate token

    IF candidate token equals observed token:
        Record recovered timestamp
        STOP SEARCH

Regenerate token using recovered timestamp

Verify regenerated token equals observed token

OUTPUT:
    Number of candidates tested
    Recovered timestamp
    Verification result

## Experimental Results — Part B

### Token Generation

| Parameter | Result |
|---|---|
| Original Unix timestamp | 1794600627 |
| Token size | 16 bytes |
| Generated token | `5aaa2ab235421d3bea1ec7a586770d98` |

### Timestamp Prediction Attack

| Parameter | Result |
|---|---|
| Attacker's estimated timestamp | 1794600747 |
| Search interval | 1794600447 to 1794601047 |
| Search window | ±300 seconds |
| Total possible timestamps | 601 |
| Candidates tested | 181 |
| Recovered timestamp | 1794600627 |
| Regenerated token | `5aaa2ab235421d3bea1ec7a586770d98` |
| Timestamp recovery | Successful |
| Token verification | Successful |

### Observations

1. The attacker did not know the exact token generation timestamp.
2. The estimated timestamp was 120 seconds later than the actual timestamp.
3. The original timestamp was recovered after testing 181 candidates.
4. The regenerated token matched the observed token exactly.
5. This demonstrates the vulnerability of timestamp-based seeding with a predictable PRNG.

In [5]:

# PART B: PREDICTABLE PASSWORD RESET TOKEN ATTACK

import time

# ------------------------------------------------------------
# 1. RESET TOKEN GENERATION
# ------------------------------------------------------------

def generate_reset_token(timestamp):
    """
    Generate a 16-byte reset token using MT19937
    seeded with a Unix timestamp.
    """
    bit_generator = np.random.MT19937(timestamp)
    rng = np.random.Generator(bit_generator)

    token = rng.integers(
        0, 256, size=16, dtype=np.uint8
    )

    return token.tobytes()


# ------------------------------------------------------------
# 2. TIMESTAMP PREDICTION ATTACK
# ------------------------------------------------------------

def predict_reset_token(observed_token, estimated_timestamp, window=300):
    """
    Search timestamps within estimated_timestamp +/- window
    to recover the seed used for token generation.
    """

    candidates_tested = 0
    recovered_timestamp = None

    start_timestamp = estimated_timestamp - window
    end_timestamp = estimated_timestamp + window

    for candidate in range(start_timestamp, end_timestamp + 1):

        candidates_tested += 1

        candidate_token = generate_reset_token(candidate)

        if candidate_token == observed_token:
            recovered_timestamp = candidate
            break

    return recovered_timestamp, candidates_tested


# ------------------------------------------------------------
# 3. GENERATE SIMULATED RESET TOKEN
# ------------------------------------------------------------

# Record actual Unix timestamp
original_timestamp = int(time.time())

# Generate token using actual timestamp
original_token = generate_reset_token(original_timestamp)

print("=== TOKEN GENERATION ===")
print("Original timestamp:", original_timestamp)
print("Generated token:", original_token.hex())


# ------------------------------------------------------------
# 4. SIMULATE ATTACKER'S APPROXIMATE TIME ESTIMATE
# ------------------------------------------------------------

window = 300

# Attacker estimates generation time to be 120 seconds later
attacker_estimate = original_timestamp + 120

start_timestamp = attacker_estimate - window
end_timestamp = attacker_estimate + window

print("\n=== TIMESTAMP PREDICTION ATTACK ===")
print("Attacker's estimated timestamp:", attacker_estimate)
print("Search interval:", start_timestamp, "to", end_timestamp)
print("Total possible candidates:", 2 * window + 1)


# ------------------------------------------------------------
# 5. BRUTE-FORCE TIMESTAMP SEARCH
# ------------------------------------------------------------

recovered_timestamp, candidates_tested = predict_reset_token(
    original_token,
    attacker_estimate,
    window
)

print("Candidates tested:", candidates_tested)
print("Recovered timestamp:", recovered_timestamp)


# ------------------------------------------------------------
# 6. VERIFY RECOVERED TIMESTAMP AND TOKEN
# ------------------------------------------------------------

if recovered_timestamp is not None:

    recovered_token = generate_reset_token(recovered_timestamp)

    print("\n=== VERIFICATION ===")
    print("Regenerated token:", recovered_token.hex())

    print(
        "Timestamp recovered correctly:",
        recovered_timestamp == original_timestamp
    )

    print(
        "Token verification successful:",
        recovered_token == original_token
    )

else:
    print("\nNo matching timestamp found.")


=== TOKEN GENERATION ===
Original timestamp: 1794600627
Generated token: 5aaa2ab235421d3bea1ec7a586770d98

=== TIMESTAMP PREDICTION ATTACK ===
Attacker's estimated timestamp: 1794600747
Search interval: 1794600447 to 1794601047
Total possible candidates: 601
Candidates tested: 181
Recovered timestamp: 1794600627

=== VERIFICATION ===
Regenerated token: 5aaa2ab235421d3bea1ec7a586770d98
Timestamp recovered correctly: True
Token verification successful: True


# Overall Conclusion

This assignment demonstrated the weaknesses of using MT19937, a statistical pseudorandom number generator, for cryptographic applications.

In Part A, a toy stream cipher was implemented using MT19937-generated keystream bytes and XOR encryption. A known-plaintext attack was performed using 14 consecutive 'A' characters. By testing all 65,536 possible 16-bit seeds, the original seed and complete plaintext were successfully recovered.

In Part B, a simulated password reset token was generated using MT19937 seeded with the current Unix timestamp. By searching timestamps within a range of ±300 seconds, the original timestamp was recovered and the generated token was reproduced successfully.

Both experiments demonstrate that deterministic pseudorandom number generators can become vulnerable when their seeds are weak or predictable.

Therefore, MT19937 should not be used for encryption keystreams, password reset tokens, authentication tokens, or other security-sensitive applications. Cryptographic applications must use cryptographically secure pseudorandom number generators (CSPRNGs).

## Experimental Results — Part B

### Token Generation

| Parameter | Result |
|---|---|
| Original Unix timestamp | 1794600496 |
| Token size | 16 bytes |
| Generated token | `dc94bea8152ab0ea2bde8e56fb958ce4` |

### Timestamp Prediction Attack

| Parameter | Result |
|---|---|
| Attacker's estimated timestamp | 1794600616 |
| Search interval | 1794600316 to 1794600916 |
| Search window | ±300 seconds |
| Total possible timestamps | 601 |
| Candidates tested | 181 |
| Recovered timestamp | 1794600496 |
| Regenerated token | `dc94bea8152ab0ea2bde8e56fb958ce4` |
| Timestamp recovery | Successful |
| Token verification | Successful |

### Observations

1. The attacker did not know the exact token generation timestamp.
2. The estimated timestamp was 120 seconds later than the actual timestamp.
3. The original timestamp was recovered after testing 181 candidates.
4. The regenerated token matched the observed token exactly.
5. This demonstrates the vulnerability of timestamp-based seeding with a predictable PRNG.